# Badini GEC — ByT5 Pilot (≈5k pairs)
Week-3 end-to-end pilot from the proposal: clean text → synthetic errors → fine-tune ByT5-base → evaluate on the mini-benchmark.

**Runtime:** GPU (T4 is fine for the pilot; A100 for full training).

In [ ]:
# 1. Setup — clone your repo and install deps
!git clone https://github.com/YOUR_USERNAME/badini-gec.git
%cd badini-gec
!pip -q install transformers datasets accelerate sentencepiece

In [ ]:
# 2. Get clean Kurmanji/Badini text from KurCorpus and filter to Arabic script
from datasets import load_dataset
ds = load_dataset('abdulhade/Kurdishcorpus', split='train', streaming=True)
# TODO: adjust field/config names after inspecting the dataset card
with open('data/raw/kmr_sample.txt', 'w', encoding='utf-8') as f:
    n = 0
    for ex in ds:
        text = ex.get('text', '')
        for line in text.split('\n'):
            f.write(line.strip() + '\n')
            n += 1
        if n > 200_000:  # enough for the pilot
            break
!python scripts/check_corpus.py data/raw/kmr_sample.txt --out data/clean/badini_pilot.txt

In [ ]:
# 3. Generate ~5k synthetic pairs
!python -m badini_gec.error_inject data/clean/badini_pilot.txt data/pairs_pilot.tsv --pairs 5000

In [ ]:
# 4. Fine-tune ByT5-base
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

MODEL = 'google/byt5-base'  # drop to byt5-small if T4 memory is tight
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

rows = [l.rstrip('\n').split('\t') for l in open('data/pairs_pilot.tsv', encoding='utf-8')]
ds = Dataset.from_dict({'src': [r[0] for r in rows], 'tgt': [r[1] for r in rows]})
ds = ds.train_test_split(test_size=0.05, seed=13)

def prep(batch):
    x = tok(batch['src'], max_length=256, truncation=True)
    y = tok(text_target=batch['tgt'], max_length=256, truncation=True)
    x['labels'] = y['input_ids']
    return x

ds = ds.map(prep, batched=True, remove_columns=['src', 'tgt'])

args = Seq2SeqTrainingArguments(
    output_dir='byt5-badini-pilot',
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    num_train_epochs=3,
    eval_strategy='epoch',
    save_strategy='epoch',
    fp16=False, bf16=torch.cuda.is_bf16_supported(),
    predict_with_generate=True,
    logging_steps=50,
)
trainer = Seq2SeqTrainer(model=model, args=args,
                         train_dataset=ds['train'], eval_dataset=ds['test'],
                         data_collator=DataCollatorForSeq2Seq(tok, model=model))
trainer.train()

In [ ]:
# 5. Try it
def correct(text):
    ids = tok(text, return_tensors='pt').input_ids.to(model.device)
    out = model.generate(ids, max_length=256, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True)

print(correct('كوردي'))  # expect script-normalized correction

In [ ]:
# 6. Evaluate on the mini-benchmark (once data/benchmark/mini.tsv exists)
import os
if os.path.exists('data/benchmark/mini.tsv'):
    rows = [l.rstrip('\n').split('\t') for l in open('data/benchmark/mini.tsv', encoding='utf-8')]
    src = [r[0] for r in rows]; ref = [r[1] for r in rows]
    hyp = [correct(s) for s in src]
    open('src.txt','w').write('\n'.join(src))
    open('hyp.txt','w').write('\n'.join(hyp))
    open('ref.txt','w').write('\n'.join(ref))
    !python -m eval.evaluate src.txt hyp.txt ref.txt
else:
    print('Mini-benchmark not built yet (Week 2 task).')

In [ ]:
# 7. Save model to Drive so it survives the session
from google.colab import drive
drive.mount('/content/drive')
trainer.save_model('/content/drive/MyDrive/byt5-badini-pilot')
tok.save_pretrained('/content/drive/MyDrive/byt5-badini-pilot')